
## Tuần 6: Huấn Luyện Đầy Đủ Mô Hình Baseline B0 (YOLOv8n)

### 1. Mục tiêu
- **Train B0 đầy đủ**: Huấn luyện hoàn chỉnh mô hình Baseline B0 phục vụ làm chuẩn đối sánh gốc cho toàn bộ nghiên cứu.
- **B0 = YOLOv8n baseline**: Kiến trúc chuẩn YOLOv8n không can thiệp SAHI hay threshold optimization.
- **Dùng split đã khóa**: Sử dụng phân vùng dữ liệu cố định từ Tuần 4 (`GroupShuffleSplit`, `seed: 41`, đã chứng minh 0 rò rỉ).
- **Dùng config đã khóa**: Toàn bộ siêu tham số đọc trực tiếp từ `configs/base.yaml` (`imgsz: 640`, `epochs: 20`, `batch: 16`, `seed: 41`).
- **Không dùng test để tuning**: Tập `test.txt` được cô lập tuyệt đối, chỉ phục vụ đánh giá khách quan cuối cùng ở giai đoạn sau.

### 2. Import thư viện
Nạp các thư viện cần thiết theo chuẩn kỹ thuật: `pathlib`, `yaml`, `json`, `shutil`, `subprocess`, `datetime`, `time`, `pandas`, `ultralytics.YOLO`.

In [7]:
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import shutil
import subprocess
import time
import pandas as pd
from ultralytics import YOLO
import yaml

def find_project_root() -> Path:
    candidate = Path.cwd().resolve()
    for parent in [candidate, *candidate.parents]:
        if (parent / "configs").is_dir() and (parent / "data").is_dir():
            return parent
    return candidate

ROOT = find_project_root()
os.chdir(ROOT)
print(f"Project root     : {ROOT}")
print(f"Working directory: {os.getcwd()}")


Project root     : /Users/Project/kltn-pcb
Working directory: /Users/Project/kltn-pcb


### 3. Load config
- Đọc `configs/base.yaml`
- Đọc `data/data.yaml`
- In toàn bộ tham số siêu tham số huấn luyện.

In [8]:
CONFIG_PATH = ROOT / "configs/base.yaml"
DATA_YAML_PATH = ROOT / "data/data.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

with open(DATA_YAML_PATH, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print("=== CẤU HÌNH HUẤN LUYỆN ĐÃ KHÓA (BASELINE B0) ===")
print(f"Model       : {base_cfg.get('model')}")
print(f"Input imgsz : {base_cfg.get('imgsz')}")
print(f"Epochs      : {base_cfg.get('epochs')}")
print(f"Batch size  : {base_cfg.get('batch')}")
print(f"Optimizer   : {base_cfg.get('optimizer')}")
print(f"Patience    : {base_cfg.get('patience')}")
print(f"Seed        : {base_cfg.get('seed')}")
print(f"Classes ({len(data_cfg.get('names', {}))}): {data_cfg.get('names')}")


=== CẤU HÌNH HUẤN LUYỆN ĐÃ KHÓA (BASELINE B0) ===
Model       : yolov8n.pt
Input imgsz : 640
Epochs      : 20
Batch size  : 16
Optimizer   : auto
Patience    : 30
Seed        : 41
Classes (6): {0: 'missing_hole', 1: 'mouse_bite', 2: 'open_circuit', 3: 'short', 4: 'spur', 5: 'spurious_copper'}


### 4. Pre-flight check
Kiểm tra nghiêm ngặt 8 điều kiện trước khi kích hoạt huấn luyện đầy đủ:
- `results/leakage_report.txt` tồn tại và có `[PASS]`
- `data/splits/train.txt`, `val.txt`, `test.txt` tồn tại
- `data/splits/split_manifest.json` tồn tại
- `runs/smoke_B0/weights/best.pt` tồn tại (bằng chứng pipeline smoke test đã pass)
- `docs/decision_log.md` đã ghi augmentation recipe đã khóa
- `data/data.yaml` đủ 6 class.

In [9]:
TRAIN_TXT = ROOT / "data/splits/train.txt"
VAL_TXT = ROOT / "data/splits/val.txt"
TEST_TXT = ROOT / "data/splits/test.txt"
MANIFEST_JSON = ROOT / "data/splits/split_manifest.json"
LEAKAGE_REPORT = ROOT / "results/leakage_report.txt"
SMOKE_BEST_PT = ROOT / "runs/smoke_B0/weights/best.pt"
DECISION_LOG = ROOT / "docs/decision_log.md"

# 1. Kiểm tra split files
assert TRAIN_TXT.is_file(), f"Thiếu file: {TRAIN_TXT}"
assert VAL_TXT.is_file(), f"Thiếu file: {VAL_TXT}"
assert TEST_TXT.is_file(), f"Thiếu file: {TEST_TXT}"
assert MANIFEST_JSON.is_file(), f"Thiếu file: {MANIFEST_JSON}"
print("[PASS] Đầy đủ các file phân vùng train/val/test và split_manifest.json.")

# 2. Kiểm tra leakage report
assert LEAKAGE_REPORT.is_file(), f"Thiếu file: {LEAKAGE_REPORT}"
leakage_content = LEAKAGE_REPORT.read_text(encoding="utf-8")
assert "[PASS]" in leakage_content, "Leakage report không đạt trạng thái PASS!"
print("[PASS] Báo cáo rò rỉ dữ liệu (leakage_report.txt) đạt PASS toàn diện.")

# 3. Kiểm tra smoke checkpoint từ Tuần 5
assert SMOKE_BEST_PT.is_file(), f"Thiếu checkpoint smoke test: {SMOKE_BEST_PT}"
print(f"[PASS] Checkpoint smoke test hợp lệ ({SMOKE_BEST_PT.relative_to(ROOT)}).")

# 4. Kiểm tra decision log đã ghi nhận augmentation recipe
assert DECISION_LOG.is_file(), f"Thiếu file: {DECISION_LOG}"
log_text = DECISION_LOG.read_text(encoding="utf-8")
assert "Augmentation Recipe" in log_text or "augmentation recipe" in log_text, "Chưa ghi nhận Augmentation Recipe trong decision_log.md!"
print("[PASS] Quyết định khóa Augmentation Recipe đã được lưu trong decision_log.md.")

# 5. Kiểm tra data.yaml đủ 6 class
classes = data_cfg.get("names", {})
assert len(classes) == 6, f"Cần 6 class, hiện có: {len(classes)}"
print(f"[PASS] data/data.yaml cấu hình chuẩn xác {len(classes)} classes.")


[PASS] Đầy đủ các file phân vùng train/val/test và split_manifest.json.
[PASS] Báo cáo rò rỉ dữ liệu (leakage_report.txt) đạt PASS toàn diện.
[PASS] Checkpoint smoke test hợp lệ (runs/smoke_B0/weights/best.pt).
[PASS] Quyết định khóa Augmentation Recipe đã được lưu trong decision_log.md.
[PASS] data/data.yaml cấu hình chuẩn xác 6 classes.


### 5. Thiết lập run
- `run_name = "B0"`
- `run_dir = runs/B0`
- `result_dir = results/B0`
- Lưu `config_used.yaml` vào cả `runs/B0` và `results/B0`
- Lưu `git_commit.txt` vào cả `runs/B0` và `results/B0`
- Tạo `run_metadata.json` có `start_time`.

In [10]:
run_name = "B0"
RUNS_DIR = ROOT / "runs" / run_name
RESULTS_DIR = ROOT / "results" / run_name
PREDS_DIR = RESULTS_DIR / "predictions"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)

try:
    commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True, cwd=ROOT).strip()
except Exception:
    commit_hash = "uncommitted_workspace"

start_dt = datetime.now(timezone.utc)
initial_run_meta = {
    "experiment": run_name,
    "mode": "full_B0",
    "model": base_cfg.get("model"),
    "imgsz": base_cfg.get("imgsz"),
    "epochs": base_cfg.get("epochs"),
    "batch": base_cfg.get("batch"),
    "seed": base_cfg.get("seed"),
    "optimizer": base_cfg.get("optimizer"),
    "patience": base_cfg.get("patience"),
    "git_commit": commit_hash,
    "start_time": start_dt.isoformat(),
}

# Lưu các file metadata vào cả runs/B0 và results/B0
for target_dir in [RUNS_DIR, RESULTS_DIR]:
    with open(target_dir / "config_used.yaml", "w", encoding="utf-8") as f:
        yaml.dump(base_cfg, f, sort_keys=False)
    (target_dir / "git_commit.txt").write_text(commit_hash + "\n", encoding="utf-8")
    with open(target_dir / "run_metadata.json", "w", encoding="utf-8") as f:
        json.dump(initial_run_meta, f, indent=2)

print(f"Đã khởi tạo thư mục thực nghiệm tại: {RUNS_DIR.relative_to(ROOT)} và {RESULTS_DIR.relative_to(ROOT)}")
print(f"Git commit hash : {commit_hash[:7]}")
print(f"Start time      : {start_dt.isoformat()}")


Đã khởi tạo thư mục thực nghiệm tại: runs/B0 và results/B0
Git commit hash : 31667de
Start time      : 2026-09-22T08:41:46.531843+00:00


### 6. Dùng augmentation recipe đã khóa
Khai báo lại chính xác bộ thông số Augmentation Recipe đã được kiểm chứng và khóa từ Tuần 5:
- `degrees: 0.0` (không xoay góc tự do)
- `shear: 0.0` (không kéo xiên)
- `perspective: 0.0` (không biến dạng phối cảnh)
- `flipud: 0.0` (không lật dọc)
- `fliplr: 0.5` (lật ngang hợp lệ)
- Các tham số `mosaic`, `hsv`, `scale`, `translate` giống hệt Tuần 5.

In [11]:
aug_recipe = {
    "degrees": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "scale": 0.5,
    "translate": 0.1,
}

print("=== AUGMENTATION RECIPE ĐÃ KHÓA CHO B0 ===")
for k, v in aug_recipe.items():
    print(f"  - {k:<12}: {v}")


=== AUGMENTATION RECIPE ĐÃ KHÓA CHO B0 ===
  - degrees     : 0.0
  - shear       : 0.0
  - perspective : 0.0
  - flipud      : 0.0
  - fliplr      : 0.5
  - mosaic      : 1.0
  - hsv_h       : 0.015
  - hsv_s       : 0.7
  - hsv_v       : 0.4
  - scale       : 0.5
  - translate   : 0.1


### 7. Train full B0
- Khởi tạo `model = YOLO(config["model"])`
- Nạp `data = data/data.yaml`
- `epochs = config["epochs"]` (20 epochs)
- `imgsz = config["imgsz"]` (640 px)
- `batch = config["batch"]` (16)
- `seed = config["seed"]` (41)
- `optimizer = config["optimizer"]` (auto)
- `patience = config["patience"]` (30)
- `project = runs`, `name = B0`, `exist_ok = True`
- Truyền toàn bộ `aug_recipe` đã khóa vào quá trình huấn luyện.

In [12]:
model = YOLO(base_cfg.get("model", "yolov8n.pt"))
start_train_time = time.time()

print("=== BẮT ĐẦU HUẤN LUYỆN FULL BASELINE B0 ===")
train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=int(base_cfg.get("epochs", 20)),
    imgsz=int(base_cfg.get("imgsz", 640)),
    batch=int(base_cfg.get("batch", 16)),
    seed=int(base_cfg.get("seed", 41)),
    optimizer=str(base_cfg.get("optimizer", "auto")),
    patience=int(base_cfg.get("patience", 30)),
    project=str(ROOT / "runs"),
    name="B0",
    exist_ok=True,
    degrees=aug_recipe["degrees"],
    shear=aug_recipe["shear"],
    perspective=aug_recipe["perspective"],
    flipud=aug_recipe["flipud"],
    fliplr=aug_recipe["fliplr"],
    mosaic=aug_recipe["mosaic"],
    hsv_h=aug_recipe["hsv_h"],
    hsv_s=aug_recipe["hsv_s"],
    hsv_v=aug_recipe["hsv_v"],
    scale=aug_recipe["scale"],
    translate=aug_recipe["translate"],
    verbose=True,
)

elapsed_train_sec = time.time() - start_train_time
print(f"\nHoàn tất huấn luyện B0 trong: {elapsed_train_sec:.2f} giây (~{elapsed_train_sec/3600:.2f} giờ)")


=== BẮT ĐẦU HUẤN LUYỆN FULL BASELINE B0 ===
New https://pypi.org/project/ultralytics/8.4.158 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.153 🚀 Python-3.13.5 torch-2.14.0 CPU (Apple M1 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/Project/kltn-pcb/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, 

### 8. Validation sau train
Đánh giá mô hình trên tập validation chuẩn (`data/splits/val.txt`) với độ phân giải `imgsz: 640`:
`model.val(data="data/data.yaml", split="val")`.

In [13]:
print("=== CHẠY VALIDATION TRÊN TẬP VAL ===")
val_results = model.val(
    data=str(DATA_YAML_PATH),
    split="val",
    imgsz=int(base_cfg.get("imgsz", 640)),
    verbose=True
)

print(f"Validation mAP@0.5     : {val_results.box.map50:.4f}")
print(f"Validation mAP@0.5:0.95: {val_results.box.map:.4f}")


=== CHẠY VALIDATION TRÊN TẬP VAL ===
Ultralytics 8.4.153 🚀 Python-3.13.5 torch-2.14.0 CPU (Apple M1 Pro)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3648.9±2006.7 MB/s, size: 968.1 KB)
val: Scanning data/raw/PKU-Market-PCB(Data enhanced version)/train/labels.cache... 368 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 368/368 118.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0s/it 1:082.8s5s
                   all        368       1860      0.765      0.636      0.645      0.269
          missing_hole         60        300      0.665       0.76      0.644      0.161
            mouse_bite         60        304      0.933      0.599      0.694      0.335
          open_circuit         64        324      0.902      0.599      0.755      0.343
                 short         60        304      0.704      0.618      0.616

### 9. Prediction mẫu
- Lấy vài ảnh từ `data/splits/val.txt`
- Chạy `model.predict()`
- Lưu ảnh dự đoán vào `results/B0/predictions/`.

In [14]:
with open(VAL_TXT, "r", encoding="utf-8") as f:
    val_imgs = [line.strip() for line in f if line.strip()]

val_sample_paths = [str(ROOT / p) for p in val_imgs[:5]]
preds = model.predict(
    source=val_sample_paths,
    save=True,
    project=str(RESULTS_DIR),
    name="predictions",
    exist_ok=True,
)

saved_preds = list(PREDS_DIR.glob("*.jpg"))
print(f"Đã lưu {len(saved_preds)} ảnh dự đoán vào: {PREDS_DIR.relative_to(ROOT)}")
for p in saved_preds[:5]:
    print(f"  - {p.name}")



0: 640x640 5 missing_holes, 1 spur, 45.2ms
1: 640x640 5 missing_holes, 1 spur, 45.2ms
2: 640x640 3 missing_holes, 45.2ms
3: 640x640 3 missing_holes, 1 spur, 45.2ms
4: 640x640 4 missing_holes, 1 spur, 45.2ms
Speed: 2.0ms preprocess, 45.2ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /Users/Project/kltn-pcb/results/B0/predictions
Đã lưu 5 ảnh dự đoán vào: results/B0/predictions
  - 10_missing_hole_01_1.jpg
  - 10_missing_hole_01.jpg
  - 10_missing_hole_01_2.jpg
  - 10_missing_hole_01_3.jpg
  - 10_missing_hole_02.jpg


### 10. Kiểm tra output & Trích xuất Epoch tốt nhất
- Kiểm tra các deliverables sau huấn luyện: `results.csv`, `weights/best.pt`, `weights/last.pt`, `confusion_matrix.png`
- Trích xuất epoch có `metrics/mAP50(B)` cao nhất từ `results.csv`.

In [15]:
results_csv_file = RUNS_DIR / "results.csv"
best_pt_file = RUNS_DIR / "weights/best.pt"
last_pt_file = RUNS_DIR / "weights/last.pt"
cm_file = RUNS_DIR / "confusion_matrix.png"

assert results_csv_file.is_file(), f"Thiếu {results_csv_file}"
assert best_pt_file.is_file(), f"Thiếu {best_pt_file}"
assert last_pt_file.is_file(), f"Thiếu {last_pt_file}"
print("[PASS] File results.csv, weights/best.pt và weights/last.pt đã tồn tại đầy đủ.")

# Kiểm tra confusion matrix
if cm_file.is_file():
    print(f"[PASS] Confusion matrix đã sinh tại: {cm_file.relative_to(ROOT)}")
else:
    print("[INFO] Confusion matrix không tự sinh.")

# Kiểm tra ảnh predictions
assert len(saved_preds) > 0, "Không có ảnh dự đoán nào trong predictions!"
print(f"[PASS] Đã xác nhận {len(saved_preds)} ảnh suy luận mẫu trong results/B0/predictions/.")


[PASS] File results.csv, weights/best.pt và weights/last.pt đã tồn tại đầy đủ.
[PASS] Confusion matrix đã sinh tại: runs/B0/confusion_matrix.png
[PASS] Đã xác nhận 5 ảnh suy luận mẫu trong results/B0/predictions/.


In [16]:
# Trích xuất epoch tốt nhất và metrics cao nhất từ results.csv
df = pd.read_csv("runs/B0/results.csv")
df.columns = df.columns.str.strip()

# Tìm epoch có mAP@0.5 cao nhất trên tập val
best_epoch_idx = df["metrics/mAP50(B)"].idxmax()
best_epoch = int(df.loc[best_epoch_idx, "epoch"])
best_map50 = df.loc[best_epoch_idx, "metrics/mAP50(B)"]
best_map50_95 = df.loc[best_epoch_idx, "metrics/mAP50-95(B)"]

print(f"Epoch tốt nhất: {best_epoch}")
print(f"Val mAP@0.5    : {best_map50:.4f}")
print(f"Val mAP@0.5:0.95: {best_map50_95:.4f}")


Epoch tốt nhất: 15
Val mAP@0.5    : 0.6755
Val mAP@0.5:0.95: 0.2406


### 11. Cập nhật run_metadata.json
Cập nhật thông tin chi tiết vào `runs/B0/run_metadata.json` và `results/B0/run_metadata.json`:
- `start_time`
- `end_time`
- `duration_seconds`
- `best_checkpoint`
- `last_checkpoint`
- `best_epoch`, `best_map50`, `best_map50_95`
- `mode = "full_B0"`.

In [17]:
end_dt = datetime.now(timezone.utc)
updated_full_meta = {
    "experiment": run_name,
    "mode": "full_B0",
    "start_time": start_dt.isoformat(),
    "end_time": end_dt.isoformat(),
    "duration_seconds": round(elapsed_train_sec, 2),
    "duration_hours": round(elapsed_train_sec / 3600.0, 2),
    "best_checkpoint": str(best_pt_file.relative_to(ROOT)),
    "last_checkpoint": str(last_pt_file.relative_to(ROOT)),
    "best_epoch": best_epoch,
    "val_mAP50": float(best_map50),
    "val_mAP50_95": float(best_map50_95),
    "git_commit": commit_hash,
    "base_model": base_cfg.get("model"),
    "seed": base_cfg.get("seed"),
    "epochs": base_cfg.get("epochs"),
    "imgsz": base_cfg.get("imgsz"),
    "batch": base_cfg.get("batch"),
    "augmentation_recipe": aug_recipe,
}

for target_dir in [RUNS_DIR, RESULTS_DIR]:
    with open(target_dir / "run_metadata.json", "w", encoding="utf-8") as f:
        json.dump(updated_full_meta, f, indent=2)

print(f"Đã cập nhật run_metadata.json tại cả {RUNS_DIR.relative_to(ROOT)} và {RESULTS_DIR.relative_to(ROOT)}:")
print(json.dumps(updated_full_meta, indent=2))


Đã cập nhật run_metadata.json tại cả runs/B0 và results/B0:
{
  "experiment": "B0",
  "mode": "full_B0",
  "start_time": "2026-09-22T08:41:46.531843+00:00",
  "end_time": "2026-09-22T17:34:14.257949+00:00",
  "duration_seconds": 31877.73,
  "duration_hours": 8.85,
  "best_checkpoint": "runs/B0/weights/best.pt",
  "last_checkpoint": "runs/B0/weights/last.pt",
  "best_epoch": 15,
  "val_mAP50": 0.67553,
  "val_mAP50_95": 0.24063,
  "git_commit": "31667dedef749a9acccca5832c47868cb39cd9e6",
  "base_model": "yolov8n.pt",
  "seed": 41,
  "epochs": 20,
  "imgsz": 640,
  "batch": 16,
  "augmentation_recipe": {
    "degrees": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "scale": 0.5,
    "translate": 0.1
  }
}


### 12. Append docs/decision_log.md
Ghi nhận toàn bộ thông số và kết quả huấn luyện B0 vào `docs/decision_log.md`:
- Ngày train B0
- Config dùng
- Seed
- Split checksum từ `split_manifest.json`
- Augmentation recipe
- Duration
- Checkpoint `best.pt`
- Epoch tốt nhất
- Ghi rõ không dùng test để tuning.

In [18]:
with open(MANIFEST_JSON, "r", encoding="utf-8") as f:
    manifest_data = json.load(f)

split_checksums = manifest_data.get("md5_checksums", {})
today_str = datetime.now().strftime("%Y-%m-%d")

log_entry = f"""
## {today_str}: Huấn Luyện Thành Công Baseline B0 (YOLOv8n Full Train)

- Quyết định: Hoàn tất huấn luyện mô hình Baseline B0 chuẩn làm mốc đối sánh (benchmark gốc).
- Cấu hình: model={base_cfg.get("model")}, imgsz={base_cfg.get("imgsz")}, epochs={base_cfg.get("epochs")}, batch={base_cfg.get("batch")}, optimizer={base_cfg.get("optimizer")}, patience={base_cfg.get("patience")}.
- Seed đã khóa: {base_cfg.get("seed")}.
- Split Checksum (MD5): train={split_checksums.get("train.txt")}, val={split_checksums.get("val.txt")}, test={split_checksums.get("test.txt")}.
- Augmentation Recipe: degrees=0.0, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=1.0, scale=0.5, translate=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4.
- Thời lượng huấn luyện (duration): {elapsed_train_sec:.2f} s (~{elapsed_train_sec/3600.0:.2f} giờ).
- Checkpoint: best checkpoint lưu tại `{best_pt_file.relative_to(ROOT)}` (Epoch tốt nhất: {best_epoch}, val mAP@0.5: {best_map50:.4f}, mAP@0.5:0.95: {best_map50_95:.4f}).
- Ràng buộc kiểm chứng: Tuyệt đối không sử dụng tập test (`data/splits/test.txt`) để tuning bất kỳ siêu tham số nào.
"""

with open(DECISION_LOG, "a", encoding="utf-8") as f:
    f.write(log_entry)

print(f"Đã append thành công kết quả huấn luyện B0 vào: {DECISION_LOG.relative_to(ROOT)}")


Đã append thành công kết quả huấn luyện B0 vào: docs/decision_log.md
